# Conversão de arquivos .txt para .csv

Este notebook converte todos os arquivos `.txt` encontrados dentro das pastas `0_raw` no workspace para o formato `.csv`. Cada arquivo de origem possui timestamp completo (`DD-MMM-YY HH:MM:SS.S`) e valor, separados por espaços, com cabeçalho e linha separadora. Os CSVs de saída são salvos na pasta `1_bronze` preservando a mesma estrutura de subpastas, com colunas `ts` (datetime completo) e `value`.

In [0]:
import os
import io
import tempfile
import pandas as pd
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Diretórios de origem (0_raw) e destino (1_bronze)
BASE_DIR = "/PrevisaoConsumoVapor/0_raw"
DEST_DIR = "/PrevisaoConsumoVapor/1_bronze"

# Conjunto de diretórios já criados para evitar chamadas repetidas à API
_created_dirs = set()

def find_txt_files(base_dir):
    """Encontra recursivamente todos os arquivos .txt dentro do diretório base."""
    txt_files = []
    for root, dirs, files in os.walk(f"/Workspace{base_dir}"):
        for f in files:
            if f.endswith('.txt'):
                txt_files.append(os.path.join(root, f))
    return sorted(txt_files)

def ensure_dir(workspace_dir):
    """Cria diretório no workspace via API se ainda não foi criado."""
    if workspace_dir not in _created_dirs:
        w.workspace.mkdirs(workspace_dir)
        _created_dirs.add(workspace_dir)

def convert_txt_to_csv(txt_path):
    """Converte um arquivo .txt para .csv preservando o timestamp completo.
    
    Formato esperado:
    - Linha 1: cabeçalho (ts / value)
    - Linha 2: separador (dashes)
    - Linhas 3+: DD-MMM-YY HH:MM:SS.S   value
    """
    # Caminho relativo preservando a estrutura de subpastas
    relative_path = os.path.relpath(txt_path, f"/Workspace{BASE_DIR}")
    csv_relative = relative_path.rsplit('.txt', 1)[0] + '.csv'
    csv_workspace_path = f"{DEST_DIR}/{csv_relative}"
    csv_fuse_path = f"/Workspace{csv_workspace_path}"
    
    # Criar diretórios de destino via API do Databricks
    dest_folder = os.path.dirname(csv_workspace_path)
    ensure_dir(dest_folder)
    
    # Ler o arquivo .txt: 3 colunas separadas por espaços (date, time, value)
    # Pular as 2 primeiras linhas (cabeçalho + separador de dashes)
    df = pd.read_csv(
        txt_path,
        sep=r'\s+',
        skiprows=2,
        header=None,
        names=['date', 'time', 'value'],
        engine='python'
    )
    
    # Combinar data + hora em um datetime completo
    df['ts'] = pd.to_datetime(
        df['date'] + ' ' + df['time'],
        format='%d-%b-%y %H:%M:%S.%f',
        errors='coerce'
    )
    df = df[['ts', 'value']].dropna(subset=['ts'])
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    
    # Escrever CSV diretamente via FUSE
    df.to_csv(csv_fuse_path, index=False)
    
    return csv_workspace_path, len(df)

# Encontrar todos os arquivos .txt
txt_files = find_txt_files(BASE_DIR)
print(f"Total de arquivos .txt encontrados: {len(txt_files)}\n")

# Converter cada arquivo
sucessos = 0
erros = 0
resultados = []

for txt_path in txt_files:
    try:
        csv_path, n_rows = convert_txt_to_csv(txt_path)
        sucessos += 1
        resultados.append((txt_path, csv_path, n_rows, 'OK'))
    except Exception as e:
        erros += 1
        resultados.append((txt_path, None, 0, str(e)))

# Exibir resumo
print(f"Convertidos com sucesso: {sucessos}")
print(f"Erros: {erros}\n")

if erros > 0:
    print("=== ERROS ===")
    for txt, csv_p, n, status in resultados:
        if status != 'OK':
            print(f"  {os.path.basename(txt)}: {status}")
    print()

print("=== ARQUIVOS CONVERTIDOS ===")
for txt, csv_p, n, status in resultados:
    if status == 'OK':
        print(f"  {os.path.basename(txt)} -> {os.path.basename(csv_p)} ({n} linhas)")